# Data Wrangling

In this example, you will learn the basics of data cleaning and munging (manipulation).
Data is the heart and soul of a data scientist and machine learner.
Therefore, it is important for you to get comfortable taking data from its natural,
messy state to a state where it is ready for machine learning algorithms.

In [ ]:
# Imports we will need for this example.

import json
import numpy
import pandas
import re

## Data: Synthetic Data Similar to the CIA World Factbook

The assignment uses the [CIA World Factbook](https://www.cia.gov/the-world-factbook/),
which contains more advanced cases than the synthetic dataset used in previous assignments.

When working with new datasets, it can be useful to create a small subset of examples to get an understanding of the data.
Then, we can iteratively add in edge cases and adapt our data wrangling code as needed to address more complex inputs.

The data can be found in this repository in the `synthetic-world-data.json` file.

Let's take a look at the data as we have in previous examples.

In [ ]:
# Our data comes in JSON, so first parse the JSON data.
with open('synthetic-world-data.json', 'r') as file:
    data = json.load(file)

# Now load the JSON data into a Pandas dataframe.
world_data = pandas.DataFrame.from_dict(data, orient = 'index')

# Sort the data by country name so it looks nice.
world_data.sort_index(axis = 0, inplace = True)

# Move the country name out of the index and into an actual column.
world_data.insert(0, 'Country', world_data.index)
world_data.reset_index(drop = True, inplace = True)

print("Our messy dataset:")
world_data

Let's also take a look at the column information:

In [ ]:
world_data.info()

Now the numerical values:

In [ ]:
world_data.describe()

Note that when we tried to get information about the numerical columns like in a previous assignment,
we didn't get nearly as much information as we did before.
If we look back to `world_data.info()`,
we can see this is because Pandas doesn't actually know which of our columns are numbers
(the dtype ("data type") for each column is just `object`).
If we look at some of the values (like "12,300,000 (2021 est.)"),
we can see why Pandas would be confused about how to treat that data.
Pandas will always try and choose a data type that can be applied to every value in the column.
So if a column has a million ints and one float,
then the column type will be float (since all ints are floats, but not all floats are ints).

Let's see if we can clean up this data!

## Initial Data Exploration (Think About It!)

Now, take a moment to look closely at the `df` DataFrame printed above and the output of `.info()`. Ask yourself:

* **Content:** What data is each column supposed to contain?
* **Data Type:** What data type (number, text, category) *should* be used?
* **Cleaning:** Which specific values look messy, incorrect, or inconsistent? Why?
* **Usefulness:** Which columns seem useful? Which seem less useful?

Thinking through these questions *before* writing code helps you build a plan for cleaning.

## Practice 1: Detect NaN Values

Looking at our data, there are multiple ways people wrote `numpy.nan`.
To a human, it is obvious that these various representations all mean the same thing.
However, it is more difficult for us machine learning folks.

Write a helper function that takes a string value and returns a boolean signaling it should be replaced with `numpy.nan`.

In [ ]:
def is_nan_value(value):
    """
    Parameters:
        value: string

    Returns bool:
        True if the value should be replaced with `numpy.nan`
        False if the value should be unchanged
    """
    # Your code here...
    pass

It is smart to test our approach as we develop.
Testing a large function that depends on many small, untested function can be hard to debug.
It is often easier to test each small function as we write them,
so we can test a large function that depends on these functions.

Let's test our solution!

In [ ]:
test_cases = [
    # (input, expected output) ...

    # numpy.nan values
    ("nan", True),
    ("NA", True),
    ("n/a", True),

    # Other values
    ("0.40% (est.)", False),
    ("5,424,860 (2022 est.)", False),
]

for i in range(len(test_cases)):
    test_case = test_cases[i]
    input_value = test_case[0]
    expected_result = test_case[1]

    actual_result = is_nan_value(input_value)
    if (actual_result is None):
        print("Go back and complete Practice 1")
        break

    assert actual_result == expected_result, "Failed case %d ('%s'): Expected: '%s', Actual: '%s'" % (i, input_value, expected_result, actual_result)

Once these test cases pass, we are more confident that our nan detection algorithm works properly.

Using our detection algorithm, the following code snippet will replace the variations of nan into a standard `numpy.nan`.

In [ ]:
for column in world_data.columns:
    for row in range(len(world_data[column])):
        value = world_data[column][row]
        if (is_nan_value(value)):
            world_data[column][row] = numpy.nan

world_data

## Practice 2: Dropping Unwanted Columns

After converting to `numpy.nan`, we can see that "A Very Sparse Column" has many nan values.

This column doesn't seem like it will be useful for us,
so let's go ahead and drop it.

To drop a column from a pandas dataframe, take a look at the following [pandas documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html).

For this practice, fill out the parameters to the `drop()` method.

In [ ]:
sparse_column_name = 'A Very Sparse Column'

# Note that pandas will raise an exception if the column we are trying to drop does not exist.
if sparse_column_name in world_data.columns:
    # Complete the function call here to drop the sparse column.
    # Your code here...
    world_data = world_data.drop()

world_data

## Practice 3: Extracting Numbers with Regex

Converting human readable nan values to machine compatible `numpy.nan` values and dropping sparse columns is a great start.
To further clean the data, we need to extract numerical values.
Our data contains the following cases:
 * Large numbers separated by commas (e.g., "5,120,000")
 * Fractional values represented as percentages (e.g., "3.1% (est.)")
 * Decimal values (e.g., "0.95")
 * Numbers with context for humans (e.g., "(2021 est.)")

In our data, it looks like many columns just contain a single main number with some potential context information around it.

(Note: The assignment contains complex cases that include multiple relevant values.)

For the problem (and in computer science in general), it can be helpful to break down a complex problem into multiple, more manageable chunks.
This process is called [problem decomposition](https://en.wikipedia.org/wiki/Decomposition_%28computer_science%29).
It helps us reason about complex problems by thinking about a specific area at a time (it also makes testing your code easier).

So, we will create multiple regular expressions to handle each case individually.

Note, there are many ways to clean data. Another approach could include manipulating the string before trying to match digits.
(E.g., we could replace commas (',') with no character ('') to make our pattern matching easier.)
Think about other ways we can use string manipulation to make our regular expressions simpler.

In [ ]:
# Build your regular expressions here.
large_number_re = r''
decimal_re = r''
percentage_re = r''

In [ ]:
# A helper function to test our regular expressions.
def test_re_matches(reg_exp, test_cases):
    if (reg_exp == r''):
        print("Go back and complete the regular expression for these test cases.")
        return

    for i in range(len(test_cases)):
        test_case = test_cases[i]
        input_value = test_case[0]
        expected_result = test_case[1]
    
        actual_result = re.search(reg_exp, input_value)
        if actual_result is not None:
            # Unpack the full match group from the Match object.
            actual_result = actual_result.group(0)

            # Normalize the match result (so testing is simpler).
            actual_result = actual_result.lower().strip()
    
        assert actual_result == expected_result, "Failed case %d ('%s'): Expected: '%s', Actual: '%s'" % (i, input_value, expected_result, actual_result)

Let's test each regular expression independently:

In [ ]:
# Large number test cases.
test_cases = [
    # (input, expected matches)...

    # Expect matches
    ("980", "980"),
    ("5,120,000", "5,120,000"),
    ("some 1,230,000 garbage", "1,230,000"),
    ("-410,680", "-410,680"),

    # Expect NOT matching
    ("", None),
    ("n/a", None),
    ("Unknown population", None),
]

test_re_matches(large_number_re, test_cases)

In [ ]:
# Decimal test cases.
test_cases = [
    # (input, expected matches)...
    ("0.54", "0.54"),
    ("10.38", "10.38"),
    ("some 0.1 garbage", "0.1"),
    ("-0.5", "-0.5"),

    # Expect NOT matching
    ("", None),
    ("n/a", None),
    ("unknown decimal", None),
]

test_re_matches(decimal_re, test_cases)

In [ ]:
# Percentage test cases.
test_cases = [
    # (input, expected matches)...
    ("54%", "54%"),
    ("0.38%", "0.38%"),
    ("some 98% garbage", "98%"),
    ("-5%", "-5%"),

    # Expect NOT matching
    ("", None),
    ("n/a", None),
    ("unknown percentage", None),
]

test_re_matches(percentage_re, test_cases)

## Practice 4: Converting Numbers After Matching Regex

Now that we can match various simple cases, we need to convert the information into an `int` or a `float`.
If we properly trim the string, we can directly convert the values into these native data types.

We provide a simple function to convert these different classes of matches.
But, the real datasets contain trickier examples that must be reasoned about, such as:
* How can we avoid detecting years?
* What should we do with cells containing multiple matches?
* Are there other anomolies that we did not account for during a test case?

Thinking about these questions can help us prepare our data even better than our initial steps.

In [ ]:
def clean_synthetic_dataframe(df):
    print(df.columns)
    for column in df.columns:
        for row_index in range(len(df[column])):
            dirty_value = df[column][row_index]

            clean_value = None
            if (re.search(percentage_re, dirty_value) is not None):
                # If we detect a percentage, convert the number into a float and adjust the decimal.
                # Your percentage conversion code here...
                pass
            elif (re.search(decimal_re, dirty_value) is not None):
                # If we detect a decimal, convert the number into a float.
                # Your decimal conversion code here...
                pass
            elif (re.search(large_number_re, dirty_value) is not None):
                # If we detect a large number, convert the number into a int.
                # Your number conversion code here...
                pass
            else:
                # If we cannot detect a number, default to numpy.nan.
                clean_value = numpy.nan

            # Apply the cleaned value back to the dataframe.
            df.loc[row_index, column] = clean_value

    return df

In [ ]:
expected_cleaned_data = {
    "A": {
        "Population": 5120000,
        "GDP Growth": 0.031,
        "Literacy": 0.95,
        "Main Exports": "oil, diamonds"
    },
    "B": {
        "Population": 12300000,
        "GDP Growth": 0.012,
        "Literacy": 0.87,
        "Main Exports": "cars, medicine"
    },
    "C": {
        "Population": 830000,
        "GDP Growth": numpy.nan,
        "Literacy": numpy.nan,
        "Main Exports": "fish"
    },
    "D": {
        "Population": 21100000,
        "GDP Growth": -0.005,
        "Literacy": 0.99,
        "Main Exports": "oil, gas"
    },
    "E": {
        "Population": 980,
        "GDP Growth": 0.025,
        "Literacy": 0.91,
        "Main Exports": "diamonds"
    }
}

expected_cleaned_df = pandas.DataFrame.from_dict(expected_cleaned_data, orient = "index")

# Add the Country column into the dataframe.
expected_cleaned_df = expected_cleaned_df.reset_index().rename(columns = {"index": "Country"})

print("Expected cleaned dataframe:")
expected_cleaned_df

In [ ]:
# Take a copy of our world data to not modify the original dataframe.
final_world_data = world_data.copy()
print("Your dataframe:")
final_world_data

In [ ]:
assert expected_cleaned_df.equals(clean_synthetic_dataframe(final_world_data))

Now, our dataframe will give us much more detail on our numeric columns:

In [ ]:
final_world_data.describe()

## Conclusion

Great! You completed the introduction examples to data wrangling.

Using the knowledge from this example, go out and clean the data on the assignment!
Remember, a great way to tackle difficult problems is to break them into manageable chunks.